In [ ]:
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np

class Mirror_descent_Banach():
    def __init__(self,s,p,q,theta,tau,gamma0,N):
        self.s = s
        self.p = p
        self.pp = p/(p-1)
        self.q = q
        self.Ln = 1
        self.fn = [np.zeros(1),np.zeros(1)]
        self.f_average_n = [np.zeros(1),np.zeros(1)]
        self.theta = theta
        self.gamma0 = gamma0 #Initial step size
        self.tau = tau #Step decay rate
        self.Num = np.zeros(1)
        self.epoch = 1 #  Current number of iterations
        self.Hn = 0
        self.a0 =  2*np.cos(1.5*math.pi*np.arccos(1-abs(0-0.5))) + (2*0)**2 - 3*abs(0-1/4)**(2/3) + 3*np.cos(5*0) - np.exp(3*0)
        self.a1 =  2*np.cos(1.5*math.pi*np.arccos(1-abs(1-0.5))) + (2*1)**2 - 3*abs(1-1/4)**(2/3) + 3*np.cos(5*1) - np.exp(3*1)
        self.N = N
        
    def cal_basis_function(self,X):
        a = min(X,1/2)
        b = min(1/2,1-X)
        Output_1 = self.integral(a,b,'sin',X)
        Output_2 = self.integral(a,b,'cos',X)
        return [Output_1,Output_2]

    def integral(self,a,b,function,X):
        if function == 'sin':
            output = np.array([0])
        elif function == 'cos':
            output = np.array([0-(-a+(-a)**2)+b-b**2-0])
        if self.Ln > 1:
            self.Num_new = 2*math.pi*self.Num[1:]
            X_List = self.Num_new*X
            X_a_List = self.Num_new*(X-a)
            X_b_List = self.Num_new*(X+b)
            inv_self_Num_new = 1/self.Num_new
            if function == 'sin':
                Output = ((-(1+2*0)*inv_self_Num_new*np.cos(X_List)+2*(inv_self_Num_new**2)*np.sin(X_List))
                          -(-(1-2*a)*inv_self_Num_new*np.cos(X_a_List)+2*(inv_self_Num_new**2)*np.sin(X_a_List)))
                Output += ((-(1-2*b)*inv_self_Num_new*np.cos(X_b_List)-2*(inv_self_Num_new**2)*np.sin(X_b_List))
                           -(-(1-2*0)*inv_self_Num_new*np.cos(X_List)-2*(inv_self_Num_new**2)*np.sin(X_List)))
            elif function == 'cos':
                Output = (((1+2*0)*inv_self_Num_new*np.sin(X_List)+2*(inv_self_Num_new**2)*np.cos(X_List))
                          -((1-2*a)*inv_self_Num_new*np.sin(X_a_List)+2*(inv_self_Num_new**2)*np.cos(X_a_List)))
                Output += (((1-2*b)*inv_self_Num_new*np.sin(X_b_List)-2*(inv_self_Num_new**2)*np.cos(X_b_List))
                           -((1-2*0)*inv_self_Num_new*np.sin(X_List)-2*(inv_self_Num_new**2)*np.cos(X_List)))
            output = np.concatenate((output,Output))
        return output

    def fn_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.fn[0]*Output[0])+np.sum(self.fn[1]*Output[1])
        
    def f_average_n_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.f_average_n[0]*Output[0])+np.sum(self.f_average_n[1]*Output[1])

    def cal_basis_function_value(self,X):
        X_list = X*np.ones(self.Ln)*(2*math.pi)*self.Num
        Output_1 = np.sin(X_list)
        Output_2 = np.cos(X_list)
        return [Output_1,Output_2]

    def fn_cal_value(self,X):
        Output = self.cal_basis_function_value(X)
        return np.sum(self.fn[0]*Output[0])+np.sum(self.fn[1]*Output[1])
        
    def f_average_n_cal_value(self,X):
        Output = self.cal_basis_function_value(X)
        return np.sum(self.f_average_n[0]*Output[0])+np.sum(self.f_average_n[1]*Output[1])

    def data_noise(self):
        c = 1.5
        W = np.random.exponential()
        U = np.random.uniform(-math.pi/2,math.pi/2)
        epsilon = 2*(np.sin(c*U)/(np.cos(U)**(1/c)))*(np.cos(U-c*U)/W)**((1-c)/c)
        return epsilon 
        
    def data(self):
        X = random.random()
        Y = self.f_rho_cal(X)
        Z = Y + random.gauss(0,0.5) #random.uniform(-0.2,0.2) #self.data_noise() #random.gauss(0,0.5)
        return (X,Z,Y)
    
    def f_rho_cal(self,x):
        a = -min(x,1/2)
        b = min(1-x,1/2)
        output = ((-(1+2*0)/(2*math.pi)*np.cos(2*math.pi*(0+x))+2/((2*math.pi)**2)*np.sin(2*math.pi*(0+x)))
                  -(-(1+2*a)/(2*math.pi)*np.cos(2*math.pi*(a+x))+2/((2*math.pi)**2)*np.sin(2*math.pi*(a+x)))
                  +(-(1-2*b)/(2*math.pi)*np.cos(2*math.pi*(b+x))-2/((2*math.pi)**2)*np.sin(2*math.pi*(b+x)))
                  -(-(1-2*0)/(2*math.pi)*np.cos(2*math.pi*(0+x))-2/((2*math.pi)**2)*np.sin(2*math.pi*(0+x))))
        output += (((1+2*0)/(8*math.pi)*np.sin(8*math.pi*(0+x))+2/((8*math.pi)**2)*np.cos(8*math.pi*(0+x)))
                   -((1+2*a)/(8*math.pi)*np.sin(8*math.pi*(a+x))+2/((8*math.pi)**2)*np.cos(8*math.pi*(a+x)))
                   +((1-2*b)/(8*math.pi)*np.sin(8*math.pi*(b+x))-2/((8*math.pi)**2)*np.cos(8*math.pi*(b+x)))
                   -((1-2*0)/(8*math.pi)*np.sin(8*math.pi*(0+x))-2/((8*math.pi)**2)*np.cos(8*math.pi*(0+x))))
        output += np.exp(0+x)*(2*0-1)-np.exp(a+x)*(2*a-1)+np.exp(b+x)*(3-2*b)-np.exp(x)*(3-2*0)
        c = np.exp(1)
        d = 1
        output -= (0-((d+(c-d)*x)*a+(c-d)*(a**2)/2+(d+(c-d)*x)*(a**2)+2*(c-d)*(a**3)/3)
                   +(d+(c-d)*x)*b+(c-d)*(b**2)/2-(d+(c-d)*x)*(b**2)-2*(c-d)*(b**3)/3)
        output += self.B(x)
        return  output
        
    def B(self,x):
        if 0 <= x <= 1/4:
            return -2*x**3 + 3/2*x**2 + 3/8*x + 1/32
        elif 1/4 < x <= 1/2:
            return -4*x**3 + 3*x**2 + 1/16
        elif 1/2 < x <= 3/4:
            return 4*x**3 - 9*x**2 + 6*x - 15/16
        elif 3/4 < x <= 1:
            return 2*x**3 - 9/2*x**2 + 21/8*x - 3/32
        else:
            return 0

    def f_rho_cal_value(self,x):
        return np.sin(2*math.pi*x)+np.cos(8*math.pi*x)+np.exp(x)-np.exp(1)*x-1*(1-x)+1.5 * np.maximum(1 - 4*np.abs(x - 0.5), 0.0)
        
    def iteration(self,X,Y):
        if self.epoch**self.theta > self.Ln:
            self.Ln += 1
            self.Num = np.append(self.Num,self.Ln-1)
            self.fn[0] = np.append(self.fn[0],0)
            self.fn[1] =  np.append(self.fn[1],0)
            self.f_average_n[0] = np.append(self.f_average_n[0],0)
            self.f_average_n[1] =  np.append(self.f_average_n[1],0)
        self.gamma = self.gamma0*(self.epoch**(-self.tau))
        fX = self.fn_cal(X)
        w_f = self.q*(abs(fX-Y)**(self.q-1))*np.sign(fX-Y)
        J_list = np.array([ j**(2*self.s*self.p) for j in range(1,self.Ln+1)])
        B = (np.sum(J_list*(np.abs(self.fn[0])**self.p))+np.sum(J_list*(np.abs(self.fn[1])**self.p))+10**(-8))**(max(0,2-self.p)/self.p)
        Output = self.cal_basis_function(X)
        Alpha = [B*(np.abs(self.fn[0])**(self.p-1))*np.sign(self.fn[0])*J_list-self.gamma*w_f*Output[0], B*(np.abs(self.fn[1])**(self.p-1))*np.sign(self.fn[1])*J_list-self.gamma*w_f*Output[1]]
        J_list_new = np.array([ j**(-2*self.s*self.pp) for j in range(1,self.Ln+1)])
        A = (np.sum(J_list_new*(np.abs(Alpha[0])**self.pp)+J_list_new*(np.abs(Alpha[1])**self.pp))+10**(-8))**(min(0,self.p-2)/self.p)
        self.fn = [A*J_list_new*(np.abs(Alpha[0])**(1/(self.p-1)))*np.sign(Alpha[0]), A*J_list_new*(np.abs(Alpha[1])**(1/(self.p-1)))*np.sign(Alpha[1])]
        if self.epoch >= self.sum_epoch/2:
            self.f_average_n[0] = self.n/(self.n+1)*self.f_average_n[0]+1/(self.n+1)*self.fn[0]
            self.f_average_n[1] = self.n/(self.n+1)*self.f_average_n[1]+1/(self.n+1)*self.fn[1]
            self.n += 1

    def test(self,test_num=1000):
        X,W = np.polynomial.legendre.leggauss(test_num)
        X = (X+1)/2
        W = W/2
        true_g = np.array([self.f_rho_cal(x) for x in X])
        true_f = np.array([self.f_rho_cal_value(x) for x in X])
        fn_g = np.array([self.fn_cal(x) for x in X])
        average_g = np.array([self.f_average_n_cal(x) for x in X])
        fn_f = np.array([self.fn_cal_value(x) for x in X])
        average_f = np.array([self.f_average_n_cal_value(x) for x in X])
        S = np.sum(W*np.abs(fn_g-true_g)**self.q)
        S1 = np.sum(W*np.abs(average_g-true_g)**self.q)
        S2 = np.sum(W*np.abs(average_f-true_f)**self.q)
        S3 = np.sum(W*np.abs(fn_f-true_f)**self.q)
        return S,S1,S2,S3

    
    def algorithm(self):
        test_num = 1000
        self.sum_epoch = self.N+1
        self.n = 0
        for i in range(1,self.sum_epoch):
            self.epoch = i
            X,Y,Z = self.data()
            self.iteration(X,Y)
        S,S1,S2,S3 = self.test(test_num)
        #print('epoch:',self.epoch,'Ln',self.Ln,S,S1,' value error ',S2,S3)
        return (S1,S)

In [ ]:
s = 0.25
p = 2.0
q = 2.0
theta = 0.35
tau = 0.5
gamma0 = 4.0
NN = [int(10**(i/5)) for i in range(1,26)]
n_repeat = 25

S_ave_all = []
S_estimator_all = []

for r in range(n_repeat):
    print("repeat =",r+1)
    S_ave = []
    S_estimator = []
    for N in NN:
        mirror_descent_Banach = Mirror_descent_Banach(s,p,q,theta,tau,gamma0,N)
        s_ave,s_estimator = mirror_descent_Banach.algorithm()
        S_ave.append(s_ave)
        S_estimator.append(s_estimator)

    S_ave_all.append(S_ave[:])
    S_estimator_all.append(S_estimator[:])

mean_S_ave = []
mean_S_estimator = []

for k in range(len(S_ave_all[0])):
    total = 0.0
    total1 = 0.0
    for r in range(n_repeat):
        total += S_ave_all[r][k]
        total1 += S_estimator_all[r][k]
    mean_S_ave.append(total/n_repeat)
    mean_S_estimator.append(total1/n_repeat)

print('Mirror_descent_Banach_ave_20 =',mean_S_ave)
print('Mirror_descent_Banach_est_20 =',mean_S_estimator)

In [ ]:
'p=1'

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np

class Mirror_descent_Banach():
    def __init__(self,s,theta,gamma0,N,L):
        self.s = s
        self.p = 1
        self.q = 1
        self.N = N
        self.theta = theta
        self.gamma0 = gamma0 #Initial step size
        self.gamma = self.gamma0 * self.N ** (-0.5)
        self.L = L
        self.Ln = math.ceil(self.N** self.theta)
        self.index = np.arange(1, self.Ln + 1, dtype=float)
        self.J_weight = self.index ** (2 * self.s)
        self.J_inverse = self.index ** (-2 * self.s)
        alpha0 = (3 * self.L / (2 * np.pi**2)) * self.index ** (-2 * self.s - 2)
        self.alpha_plus = [ alpha0.copy(), alpha0.copy()]
        self.alpha_minus = [ alpha0.copy(), alpha0.copy()]   
        self.fn = [ self.alpha_plus[0] - self.alpha_minus[0], self.alpha_plus[1] - self.alpha_minus[1]]
        self.f_average_n = [ np.zeros(self.Ln), np.zeros(self.Ln)]
        self.Num = np.arange(self.Ln)
        self.epoch = 1 #  Current number of iterations
        self.a0 =  2*np.cos(1.5*math.pi*np.arccos(1-abs(0-0.5))) + (2*0)**2 - 3*abs(0-1/4)**(2/3) + 3*np.cos(5*0) - np.exp(3*0)
        self.a1 =  2*np.cos(1.5*math.pi*np.arccos(1-abs(1-0.5))) + (2*1)**2 - 3*abs(1-1/4)**(2/3) + 3*np.cos(5*1) - np.exp(3*1)
        
    def cal_basis_function(self,X):
        a = min(X,1/2)
        b = min(1/2,1-X)
        Output_1 = self.integral(a,b,'sin',X)
        Output_2 = self.integral(a,b,'cos',X)
        return [Output_1,Output_2]

    def integral(self,a,b,function,X):
        if function == 'sin':
            output = np.array([0])
        elif function == 'cos':
            output = np.array([0-(-a+(-a)**2)+b-b**2-0])
        if self.Ln > 1:
            self.Num_new = 2*math.pi*self.Num[1:]
            X_List = self.Num_new*X
            X_a_List = self.Num_new*(X-a)
            X_b_List = self.Num_new*(X+b)
            inv_self_Num_new = 1/self.Num_new
            if function == 'sin':
                Output = ((-(1+2*0)*inv_self_Num_new*np.cos(X_List)+2*(inv_self_Num_new**2)*np.sin(X_List))
                          -(-(1-2*a)*inv_self_Num_new*np.cos(X_a_List)+2*(inv_self_Num_new**2)*np.sin(X_a_List)))
                Output += ((-(1-2*b)*inv_self_Num_new*np.cos(X_b_List)-2*(inv_self_Num_new**2)*np.sin(X_b_List))
                           -(-(1-2*0)*inv_self_Num_new*np.cos(X_List)-2*(inv_self_Num_new**2)*np.sin(X_List)))
            elif function == 'cos':
                Output = (((1+2*0)*inv_self_Num_new*np.sin(X_List)+2*(inv_self_Num_new**2)*np.cos(X_List))
                          -((1-2*a)*inv_self_Num_new*np.sin(X_a_List)+2*(inv_self_Num_new**2)*np.cos(X_a_List)))
                Output += (((1-2*b)*inv_self_Num_new*np.sin(X_b_List)-2*(inv_self_Num_new**2)*np.cos(X_b_List))
                           -((1-2*0)*inv_self_Num_new*np.sin(X_List)-2*(inv_self_Num_new**2)*np.cos(X_List)))
            output = np.concatenate((output,Output))
        return output

    def fn_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.fn[0]*Output[0])+np.sum(self.fn[1]*Output[1])
        
    def f_average_n_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.f_average_n[0]*Output[0])+np.sum(self.f_average_n[1]*Output[1])

    def cal_basis_function_value(self,X):
        X_list = X*np.ones(self.Ln)*(2*math.pi)*self.Num
        Output_1 = np.sin(X_list)
        Output_2 = np.cos(X_list)
        return [Output_1,Output_2]

    def fn_cal_value(self,X):
        Output = self.cal_basis_function_value(X)
        return np.sum(self.fn[0]*Output[0])+np.sum(self.fn[1]*Output[1])
        
    def f_average_n_cal_value(self,X):
        Output = self.cal_basis_function_value(X)
        return np.sum(self.f_average_n[0]*Output[0])+np.sum(self.f_average_n[1]*Output[1])

    def data_noise(self):
        c = 1.5
        W = np.random.exponential()
        U = np.random.uniform(-math.pi/2,math.pi/2)
        epsilon = 2*(np.sin(c*U)/(np.cos(U)**(1/c)))*(np.cos(U-c*U)/W)**((1-c)/c)
        return epsilon 
        
    def data(self):
        X = random.random()
        Y = self.f_rho_cal(X)
        Z = Y + random.gauss(0,0.5) #random.uniform(-0.2,0.2) #self.data_noise() #random.gauss(0,0.5)
        return (X,Z,Y)
    
    def f_rho_cal(self,x):
        a = -min(x,1/2)
        b = min(1-x,1/2)
        output = ((-(1+2*0)/(2*math.pi)*np.cos(2*math.pi*(0+x))+2/((2*math.pi)**2)*np.sin(2*math.pi*(0+x)))
                  -(-(1+2*a)/(2*math.pi)*np.cos(2*math.pi*(a+x))+2/((2*math.pi)**2)*np.sin(2*math.pi*(a+x)))
                  +(-(1-2*b)/(2*math.pi)*np.cos(2*math.pi*(b+x))-2/((2*math.pi)**2)*np.sin(2*math.pi*(b+x)))
                  -(-(1-2*0)/(2*math.pi)*np.cos(2*math.pi*(0+x))-2/((2*math.pi)**2)*np.sin(2*math.pi*(0+x))))
        output += (((1+2*0)/(8*math.pi)*np.sin(8*math.pi*(0+x))+2/((8*math.pi)**2)*np.cos(8*math.pi*(0+x)))
                   -((1+2*a)/(8*math.pi)*np.sin(8*math.pi*(a+x))+2/((8*math.pi)**2)*np.cos(8*math.pi*(a+x)))
                   +((1-2*b)/(8*math.pi)*np.sin(8*math.pi*(b+x))-2/((8*math.pi)**2)*np.cos(8*math.pi*(b+x)))
                   -((1-2*0)/(8*math.pi)*np.sin(8*math.pi*(0+x))-2/((8*math.pi)**2)*np.cos(8*math.pi*(0+x))))
        output += np.exp(0+x)*(2*0-1)-np.exp(a+x)*(2*a-1)+np.exp(b+x)*(3-2*b)-np.exp(x)*(3-2*0)
        c = np.exp(1)
        d = 1
        output -= (0-((d+(c-d)*x)*a+(c-d)*(a**2)/2+(d+(c-d)*x)*(a**2)+2*(c-d)*(a**3)/3)
                   +(d+(c-d)*x)*b+(c-d)*(b**2)/2-(d+(c-d)*x)*(b**2)-2*(c-d)*(b**3)/3)
        output += self.B(x)
        return  output
        
    def B(self,x):
        if 0 <= x <= 1/4:
            return -2*x**3 + 3/2*x**2 + 3/8*x + 1/32
        elif 1/4 < x <= 1/2:
            return -4*x**3 + 3*x**2 + 1/16
        elif 1/2 < x <= 3/4:
            return 4*x**3 - 9*x**2 + 6*x - 15/16
        elif 3/4 < x <= 1:
            return 2*x**3 - 9/2*x**2 + 21/8*x - 3/32
        else:
            return 0

    def f_rho_cal_value(self,x):
        return np.sin(2*math.pi*x)+np.cos(8*math.pi*x)+np.exp(x)-np.exp(1)*x-1*(1-x)+1.5 * np.maximum(1 - 4*np.abs(x - 0.5), 0.0)
        
    def iteration(self, X, Y):
        Output = self.cal_basis_function(X)
        fX = (np.sum(self.fn[0] * Output[0]) + np.sum(self.fn[1] * Output[1]))
        r_n = np.sign(fX - Y)
        exponent_sin = (self.gamma * r_n * self.J_inverse * Output[0])
        exponent_cos = (self.gamma * r_n * self.J_inverse * Output[1])
        exponent_sin = np.clip(exponent_sin, -200, 200)
        exponent_cos = np.clip(exponent_cos, -200, 200)
        alpha_plus_sin_new = self.alpha_plus[0] * np.exp(-exponent_sin)
        alpha_minus_sin_new = self.alpha_minus[0] * np.exp(exponent_sin)
        alpha_plus_cos_new = self.alpha_plus[1] * np.exp(-exponent_cos)
        alpha_minus_cos_new = self.alpha_minus[1] * np.exp(exponent_cos)
        weighted_mass = np.sum( self.J_weight * (alpha_plus_sin_new + alpha_minus_sin_new + alpha_plus_cos_new + alpha_minus_cos_new))
        Z_n = max(1.0, weighted_mass / self.L)
        self.alpha_plus = [ alpha_plus_sin_new / Z_n, alpha_plus_cos_new / Z_n ]
        self.alpha_minus = [ alpha_minus_sin_new / Z_n, alpha_minus_cos_new / Z_n ]
        self.fn = [ self.alpha_plus[0] - self.alpha_minus[0], self.alpha_plus[1] - self.alpha_minus[1] ]
        if self.epoch >= self.sum_epoch/2:
            self.f_average_n[0] = self.n/(self.n+1)*self.f_average_n[0]+1/(self.n+1)*self.fn[0]
            self.f_average_n[1] = self.n/(self.n+1)*self.f_average_n[1]+1/(self.n+1)*self.fn[1]
            self.n += 1
            
    def test(self,test_num=1000):
        X,W = np.polynomial.legendre.leggauss(test_num)
        X = (X+1)/2
        W = W/2
        true_g = np.array([self.f_rho_cal(x) for x in X])
        true_f = np.array([self.f_rho_cal_value(x) for x in X])
        fn_g = np.array([self.fn_cal(x) for x in X])
        average_g = np.array([self.f_average_n_cal(x) for x in X])
        fn_f = np.array([self.fn_cal_value(x) for x in X])
        average_f = np.array([self.f_average_n_cal_value(x) for x in X])
        S = np.sum(W*np.abs(fn_g-true_g)**self.q)
        S1 = np.sum(W*np.abs(average_g-true_g)**self.q)
        S2 = np.sum(W*np.abs(average_f-true_f)**self.q)
        S3 = np.sum(W*np.abs(fn_f-true_f)**self.q)
        return S,S1,S2,S3

    
    def algorithm(self):
        test_num = 1000
        self.sum_epoch = self.N+1
        self.n = 0
        for i in range(1,self.sum_epoch):
            self.epoch = i
            X,Y,Z = self.data()
            self.iteration(X,Y)
        S,S1,S2,S3 = self.test(test_num)
        #print('epoch:',self.epoch,'Ln',self.Ln,S,S1,' value error ',S2,S3)
        return (S1,S)

In [ ]:
s = 0.25
theta = 0.35
gamma0 = 7.5
NN = [int(10**(i/5)) for i in range(1,26)]
L = 20
n_repeat = 25

S_ave_all = []
S_estimator_all = []

for r in range(n_repeat):
    S_ave = []
    S_estimator = []
    for N in NN:
        mirror_descent_Banach = Mirror_descent_Banach(s,theta,gamma0,N,L)
        s_ave,s_estimator = mirror_descent_Banach.algorithm()
        S_ave.append(s_ave)
        S_estimator.append(s_estimator)
    S_ave_all.append(S_ave[:])
    S_estimator_all.append(S_estimator[:])
    print("repeat =",r+1)

mean_S_ave = []
mean_S_estimator = []

for k in range(len(S_ave_all[0])):
    total = 0.0
    total1 = 0.0
    for r in range(n_repeat):
        total += S_ave_all[r][k]
        total1 += S_estimator_all[r][k]
    mean_S_ave.append(total/n_repeat)
    mean_S_estimator.append(total1/n_repeat)

print('Mirror_descent_Banach_ave_10 =',mean_S_ave)
print('Mirror_descent_Banach_est_10 =',mean_S_estimator)

In [ ]:
'function : 4 * np.maximum(1 - 4*np.abs(x - 0.5), 0.0)'

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np

class Mirror_descent_Banach():
    def __init__(self,s,theta,gamma0,N,L):
        self.s = s
        self.p = 1
        self.q = 1
        self.N = N
        self.theta = theta
        self.gamma0 = gamma0 #Initial step size
        self.gamma = self.gamma0 * self.N ** (-0.5)
        self.L = L
        self.Ln = math.ceil(self.N** self.theta)
        self.index = np.arange(1, self.Ln + 1, dtype=float)
        self.J_weight = self.index ** (2 * self.s)
        self.J_inverse = self.index ** (-2 * self.s)
        alpha0 = (3 * self.L / (2 * np.pi**2)) * self.index ** (-2 * self.s - 2)
        self.alpha_plus = [ alpha0.copy(), alpha0.copy()]
        self.alpha_minus = [ alpha0.copy(), alpha0.copy()]   
        self.fn = [ self.alpha_plus[0] - self.alpha_minus[0], self.alpha_plus[1] - self.alpha_minus[1]]
        self.f_average_n = [ np.zeros(self.Ln), np.zeros(self.Ln)]
        self.Num = np.arange(self.Ln)
        self.epoch = 1 #  Current number of iterations
        self.a0 =  2*np.cos(1.5*math.pi*np.arccos(1-abs(0-0.5))) + (2*0)**2 - 3*abs(0-1/4)**(2/3) + 3*np.cos(5*0) - np.exp(3*0)
        self.a1 =  2*np.cos(1.5*math.pi*np.arccos(1-abs(1-0.5))) + (2*1)**2 - 3*abs(1-1/4)**(2/3) + 3*np.cos(5*1) - np.exp(3*1)
        
    def cal_basis_function(self,X):
        a = min(X,1/2)
        b = min(1/2,1-X)
        Output_1 = self.integral(a,b,'sin',X)
        Output_2 = self.integral(a,b,'cos',X)
        return [Output_1,Output_2]

    def integral(self,a,b,function,X):
        if function == 'sin':
            output = np.array([0])
        elif function == 'cos':
            output = np.array([0-(-a+(-a)**2)+b-b**2-0])
        if self.Ln > 1:
            self.Num_new = 2*math.pi*self.Num[1:]
            X_List = self.Num_new*X
            X_a_List = self.Num_new*(X-a)
            X_b_List = self.Num_new*(X+b)
            inv_self_Num_new = 1/self.Num_new
            if function == 'sin':
                Output = ((-(1+2*0)*inv_self_Num_new*np.cos(X_List)+2*(inv_self_Num_new**2)*np.sin(X_List))
                          -(-(1-2*a)*inv_self_Num_new*np.cos(X_a_List)+2*(inv_self_Num_new**2)*np.sin(X_a_List)))
                Output += ((-(1-2*b)*inv_self_Num_new*np.cos(X_b_List)-2*(inv_self_Num_new**2)*np.sin(X_b_List))
                           -(-(1-2*0)*inv_self_Num_new*np.cos(X_List)-2*(inv_self_Num_new**2)*np.sin(X_List)))
            elif function == 'cos':
                Output = (((1+2*0)*inv_self_Num_new*np.sin(X_List)+2*(inv_self_Num_new**2)*np.cos(X_List))
                          -((1-2*a)*inv_self_Num_new*np.sin(X_a_List)+2*(inv_self_Num_new**2)*np.cos(X_a_List)))
                Output += (((1-2*b)*inv_self_Num_new*np.sin(X_b_List)-2*(inv_self_Num_new**2)*np.cos(X_b_List))
                           -((1-2*0)*inv_self_Num_new*np.sin(X_List)-2*(inv_self_Num_new**2)*np.cos(X_List)))
            output = np.concatenate((output,Output))
        return output

    def fn_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.fn[0]*Output[0])+np.sum(self.fn[1]*Output[1])
        
    def f_average_n_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.f_average_n[0]*Output[0])+np.sum(self.f_average_n[1]*Output[1])

    def cal_basis_function_value(self,X):
        X_list = X*np.ones(self.Ln)*(2*math.pi)*self.Num
        Output_1 = np.sin(X_list)
        Output_2 = np.cos(X_list)
        return [Output_1,Output_2]

    def fn_cal_value(self,X):
        Output = self.cal_basis_function_value(X)
        return np.sum(self.fn[0]*Output[0])+np.sum(self.fn[1]*Output[1])
        
    def f_average_n_cal_value(self,X):
        Output = self.cal_basis_function_value(X)
        return np.sum(self.f_average_n[0]*Output[0])+np.sum(self.f_average_n[1]*Output[1])

    def data_noise(self):
        c = 1.5
        W = np.random.exponential()
        U = np.random.uniform(-math.pi/2,math.pi/2)
        epsilon = 2*(np.sin(c*U)/(np.cos(U)**(1/c)))*(np.cos(U-c*U)/W)**((1-c)/c)
        return epsilon 
        
    def data(self):
        X = random.random()
        Y = self.f_rho_cal(X)
        Z = Y + random.gauss(0,0.5) #random.uniform(-0.2,0.2) #self.data_noise() #random.gauss(0,0.5)
        return (X,Z,Y)
    
    def f_rho_cal(self,x):
        return  8*self.B(x)/3
        
    def B(self,x):
        if 0 <= x <= 1/4:
            return -2*x**3 + 3/2*x**2 + 3/8*x + 1/32
        elif 1/4 < x <= 1/2:
            return -4*x**3 + 3*x**2 + 1/16
        elif 1/2 < x <= 3/4:
            return 4*x**3 - 9*x**2 + 6*x - 15/16
        elif 3/4 < x <= 1:
            return 2*x**3 - 9/2*x**2 + 21/8*x - 3/32
        else:
            return 0

    def f_rho_cal_value(self,x):
        return 4 * np.maximum(1 - 4*np.abs(x - 0.5), 0.0)
        
    def iteration(self, X, Y):
        Output = self.cal_basis_function(X)
        fX = (np.sum(self.fn[0] * Output[0]) + np.sum(self.fn[1] * Output[1]))
        r_n = np.sign(fX - Y)
        exponent_sin = (self.gamma * r_n * self.J_inverse * Output[0])
        exponent_cos = (self.gamma * r_n * self.J_inverse * Output[1])
        exponent_sin = np.clip(exponent_sin, -200, 200)
        exponent_cos = np.clip(exponent_cos, -200, 200)
        alpha_plus_sin_new = self.alpha_plus[0] * np.exp(-exponent_sin)
        alpha_minus_sin_new = self.alpha_minus[0] * np.exp(exponent_sin)
        alpha_plus_cos_new = self.alpha_plus[1] * np.exp(-exponent_cos)
        alpha_minus_cos_new = self.alpha_minus[1] * np.exp(exponent_cos)
        weighted_mass = np.sum( self.J_weight * (alpha_plus_sin_new + alpha_minus_sin_new + alpha_plus_cos_new + alpha_minus_cos_new))
        Z_n = max(1.0, weighted_mass / self.L)
        self.alpha_plus = [ alpha_plus_sin_new / Z_n, alpha_plus_cos_new / Z_n ]
        self.alpha_minus = [ alpha_minus_sin_new / Z_n, alpha_minus_cos_new / Z_n ]
        self.fn = [ self.alpha_plus[0] - self.alpha_minus[0], self.alpha_plus[1] - self.alpha_minus[1] ]
        if self.epoch >= self.sum_epoch/2:
            self.f_average_n[0] = self.n/(self.n+1)*self.f_average_n[0]+1/(self.n+1)*self.fn[0]
            self.f_average_n[1] = self.n/(self.n+1)*self.f_average_n[1]+1/(self.n+1)*self.fn[1]
            self.n += 1
            
    def test(self,test_num=1000):
        X,W = np.polynomial.legendre.leggauss(test_num)
        X = (X+1)/2
        W = W/2
        true_g = np.array([self.f_rho_cal(x) for x in X])
        true_f = np.array([self.f_rho_cal_value(x) for x in X])
        fn_g = np.array([self.fn_cal(x) for x in X])
        average_g = np.array([self.f_average_n_cal(x) for x in X])
        fn_f = np.array([self.fn_cal_value(x) for x in X])
        average_f = np.array([self.f_average_n_cal_value(x) for x in X])
        S = np.sum(W*np.abs(fn_g-true_g)**self.q)
        S1 = np.sum(W*np.abs(average_g-true_g)**self.q)
        S2 = np.sum(W*np.abs(average_f-true_f)**self.q)
        S3 = np.sum(W*np.abs(fn_f-true_f)**self.q)
        return S,S1,S2,S3

    
    def algorithm(self):
        test_num = 1000
        self.sum_epoch = self.N+1
        self.n = 0
        for i in range(1,self.sum_epoch):
            self.epoch = i
            X,Y,Z = self.data()
            self.iteration(X,Y)
        S,S1,S2,S3 = self.test(test_num)
        #print('epoch:',self.epoch,'Ln',self.Ln,S,S1,' value error ',S2,S3)
        return (S1,S)

In [ ]:
s = 0.25
theta = 0.35
gamma0 = 5.0
NN = [int(10**(i/5)) for i in range(1,26)]
L = 20
n_repeat = 25

S_ave_all = []
S_estimator_all = []

for r in range(n_repeat):
    S_ave = []
    S_estimator = []
    for N in NN:
        mirror_descent_Banach = Mirror_descent_Banach(s,theta,gamma0,N,L)
        s_ave,s_estimator = mirror_descent_Banach.algorithm()
        S_ave.append(s_ave)
        S_estimator.append(s_estimator)
    S_ave_all.append(S_ave[:])
    S_estimator_all.append(S_estimator[:])
    print("repeat =",r+1)

mean_S_ave = []
mean_S_estimator = []

for k in range(len(S_ave_all[0])):
    total = 0.0
    total1 = 0.0
    for r in range(n_repeat):
        total += S_ave_all[r][k]
        total1 += S_estimator_all[r][k]
    mean_S_ave.append(total/n_repeat)
    mean_S_estimator.append(total1/n_repeat)

print('Mirror_descent_Banach_ave_10_ban =',mean_S_ave)
print('Mirror_descent_Banach_est_10_ban =',mean_S_estimator)